# Dynamic init creation
> Dynamically create init when interfacing estimators from other packages

- toc: false
- badges: true
- comments: true
- categories: [software-design]

## Preliminaries

In [ ]:
!pip install sktime fbprophet

## Motivation
The main goal of sktime is to develop a unified framework for machine
learning with time series. We do this by creating a common interface for
different types of algorithms. Instead of re-implementing algorithms from
scratch, we try to interface existing algorithm implementations whenever
possible and merely expose them through a common interface. This often
requires writing the constructor (`__init__.py`) with the appropriate
arguments from the algorithm that we want to interface.

In this blog post, I discuss how we can create the constructor automatically
 based on the constructor of the interfaced algorithm and any additional
 arguments we may want to add.

I saw the idea originally in the [HCrystalball package](https://github.com/heidelbergcement/hcrystalball/blob/master/src/hcrystalball/wrappers/_base.py), but cleaned up the functions a little bit to make them more
readable.

The idea relies on dynamic function creation, you can find a good discussion
 of that topic in this [blog post](https://philip-trauner.me/blog/post/python-tips-dynamic-function-definition).

## Writing a decorator for dynamic init creation

In [5]:
import inspect
from types import FunctionType
from fbprophet import Prophet
from sktime.forecasting.base._base import BaseForecaster

def _get_param_dict(signature):
    return {
        p.name: p.default if p.default != inspect.Parameter.empty else None
        for p in signature.parameters.values()
        if p.name != "self" and p.kind != p.VAR_KEYWORD and p.kind != p.VAR_POSITIONAL
    }

def _get_params(func):
    signature = inspect.signature(func)
    return _get_param_dict(signature)

def create_init(model_cls):
    """Decorator to dynamically create init function based on wrapped `model_cls`"""

    def new_init(base_init):
        # combine params from wrapper class and wrapped model
        params = _get_params(model_cls.__init__)
        params.update(_get_params(base_init))

        # compile function code from string representation
        assignments = "; ".join([f"self.{name} = {name}" for name in params.keys()])
        string = f'def __init__(self, {", ".join(params.keys())}): {assignments}'
        code = compile(string, "<string>", "exec")

        return FunctionType(code.co_consts[0], base_init.__globals__, name="__init__", argdefs=tuple(params.values()))

    return new_init

## Example
In this example, we use [`fbprophet`](https://facebook.github.io/prophet/).

In [12]:
# create new class using dynamic init creator
class ProphetForecaster(BaseForecaster):
    """Test docstring"""

    @create_init(Prophet)
    def __init__(foo=1, bar=2):
        pass

f = ProphetForecaster()
f.get_params()

{'bar': 2,
 'changepoint_prior_scale': 0.05,
 'changepoint_range': 0.8,
 'changepoints': None,
 'daily_seasonality': 'auto',
 'foo': 1,
 'growth': 'linear',
 'holidays': None,
 'holidays_prior_scale': 10.0,
 'interval_width': 0.8,
 'mcmc_samples': 0,
 'n_changepoints': 25,
 'seasonality_mode': 'additive',
 'seasonality_prior_scale': 10.0,
 'stan_backend': None,
 'uncertainty_samples': 1000,
 'weekly_seasonality': 'auto',
 'yearly_seasonality': 'auto'}

## Future work

A few further considerations:
* It would be preferable to use the `CodeType` `replace` method, but it's only
available from Python >=3.8, see https://bugs.python.org/issue37032
* Ideally, we would want to add minor extensions to unify parameter names (e.g. `sp` for seasonality, `n_jobs` for multiprocessing)?
* Can we automatically update the docstring too?